# TP 01 : Analyse de Variance (ANOVA) à Deux Facteurs Croisés
**Master 2 Biochimie Appliquée — Université M'Hamed Bougara de Boumerdès (UMBB)**  
*Enseignante : Dr. Sarra BENMOUMOU (Ph.D.)*

---

## Contexte Biologique :
Nous étudions l'effet hépatoprotecteur d'une molécule antioxydante naturelle (polyphénol) administrée à deux doses (50 et 100 mg/kg) vs Véhicule témoin, chez des souris saines (WT) et des souris modèles du diabète de type 2 (`db_db`).  
La variable d'intérêt est le taux hépatique de **Malondialdéhyde (MDA)**, un marqueur clé de la péroxydation lipidique membranaire.

## Objectifs :
1. Tester l'effet principal du Traitement, du Génotype et surtout leur **Interaction**.
2. Réaliser le tracé du profil d'interaction (`interaction_plot`).
3. Conclure biologiquement sur la spécificité thérapeutique du polyphénol.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Chargement des données
df = pd.read_csv('../datasets/peroxydation_mda_diabete.csv')
display(df.head())


## 1. Tracé du Profil d'Interaction
Les profils sont-ils parallèles (additivité) ou sécants (interaction) ?


In [ ]:
plt.figure(figsize=(8, 5))

# Calcul des moyennes pour le tracé d'interaction
mean_data = df.groupby(['Traitement', 'Genotype'])['MDA_nmol_mg_prot'].mean().reset_index()

sns.lineplot(
    data=mean_data,
    x='Traitement',
    y='MDA_nmol_mg_prot',
    hue='Genotype',
    marker='o',
    markersize=9,
    linewidth=2.5,
    palette={'WT': '#1A365D', 'db_db': '#D97706'}
)

plt.title("Profil d'Interaction : Traitement x Génotype", fontsize=14, fontweight='bold')
plt.xlabel("Traitement", fontweight='bold')
plt.ylabel("MDA Moyen (nmol/mg)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 2. Table ANOVA à Deux Facteurs Croisés


In [ ]:
# Calcul complet de l'ANOVA à deux facteurs
N = len(df)
grand_mean = df['MDA_nmol_mg_prot'].mean()

I = df['Traitement'].nunique()
J = df['Genotype'].nunique()
K = N // (I * J)

SS_total = ((df['MDA_nmol_mg_prot'] - grand_mean)**2).sum()

means_A = df.groupby('Traitement')['MDA_nmol_mg_prot'].mean()
SS_A = (J * K) * ((means_A - grand_mean)**2).sum()

means_B = df.groupby('Genotype')['MDA_nmol_mg_prot'].mean()
SS_B = (I * K) * ((means_B - grand_mean)**2).sum()

means_AB = df.groupby(['Traitement', 'Genotype'])['MDA_nmol_mg_prot'].mean()
SS_cell = K * ((means_AB - grand_mean)**2).sum()
SS_AB = SS_cell - SS_A - SS_B

SS_res = SS_total - SS_cell

df_A = I - 1
df_B = J - 1
df_AB = (I - 1) * (J - 1)
df_res = N - (I * J)

MS_A = SS_A / df_A
MS_B = SS_B / df_B
MS_AB = SS_AB / df_AB
MS_res = SS_res / df_res

F_A = MS_A / MS_res
F_B = MS_B / MS_res
F_AB = MS_AB / MS_res

p_A = 1 - stats.f.cdf(F_A, df_A, df_res)
p_B = 1 - stats.f.cdf(F_B, df_B, df_res)
p_AB = 1 - stats.f.cdf(F_AB, df_AB, df_res)

table_anova = pd.DataFrame({
    'Source': ['Traitement (A)', 'Genotype (B)', 'Interaction A x B', 'Résiduelle'],
    'ddl': [df_A, df_B, df_AB, df_res],
    'SS': [round(SS_A, 2), round(SS_B, 2), round(SS_AB, 2), round(SS_res, 2)],
    'MS': [round(MS_A, 2), round(MS_B, 2), round(MS_AB, 2), round(MS_res, 2)],
    'F_obs': [round(F_A, 2), round(F_B, 2), round(F_AB, 2), np.nan],
    'p_value': [f'{p_A:.4e}', f'{p_B:.4e}', f'{p_AB:.4e}', np.nan]
})

print("=== TABLE ANOVA A DEUX FACTEURS CROISES ===")
display(table_anova)


## 3. Interprétation Biologique des Résultats :
- **L'interaction $A 	imes B$ est-elle statistiquement significative ?**
- Pourquoi le polyphénol est-il particulièrement intéressant pour la prise en charge des complications diabétiques ?
